# Deep Learning 1 &mdash; Assignment 4

Fourth assignment for the 2025 Deep Learning 1 course (NWI-IMC070A) of the Radboud University.

-----

**Names:** Nele Haferkorn

**Group:** 13

-----

**Instructions:**
* Fill in your names and the name of your group.
* Answer the questions and complete the code where necessary.
* Keep your answers brief, one or two sentences is usually enough.
* Re-run the whole notebook before you submit your work.
* Save the notebook as a PDF and submit that in Brightspace together with the `.ipynb` notebook file.
* The easiest way to make a PDF of your notebook is via File > Print Preview and then use your browser's print option to print to PDF.

## Objectives

In this assignment you will
1. Experiment with convolutional neural networks
2. Train a convolutional neural network on a speech dataset
3. Investigate the effect of dropout and batch normalization
4. Define and train a residual neural network

## Required software

If you haven't done so already, you will need to install the following additional libraries:
* `torch` and `torchvision` for PyTorch,
* `python_speech_features` to compute MFCC features.

All libraries can be installed with `pip install`.

In [ ]:
%matplotlib inline
import os
import numpy as np
import matplotlib.pyplot as plt
import torch
import time
from scipy.io import wavfile
from IPython import display

# Fix the seed, so outputs are exactly reproducible
torch.manual_seed(42)

# Use the GPU if available
def detect_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    elif torch.backends.mps.is_available():
        return torch.device("mps")
    else:
        return torch.device("cpu")
device = detect_device()

## 4.1 Convolution and receptive fields (9 points)

We will first define some helper functions to plot the receptive field of a node in a network.

In [ ]:
def show_image(img, title=None, new_figure=True):
    if new_figure:
        plt.figure(figsize=(5, 5))
    im = plt.imshow(img, interpolation='none', aspect='equal', cmap='gray')
    ax = plt.gca();

    # plot pixel numbers and grid lines
    ax.set_xticks(np.arange(0, img.shape[1], 1))
    ax.set_yticks(np.arange(0, img.shape[0], 1))
    ax.set_xticklabels(np.arange(0, img.shape[1], 1))
    ax.set_yticklabels(np.arange(0, img.shape[0], 1))
    ax.set_xticks(np.arange(-.5, img.shape[1], 1), minor=True)
    ax.set_yticks(np.arange(-.5, img.shape[0], 1), minor=True)
    ax.grid(which='minor', color='gray', linestyle='-', linewidth=1.5)

    # hide axis outline
    for spine in ax.spines.values():
        spine.set_visible(False)

    if title is not None:
        plt.title(title)

# set all weights in the network to one,
# all biases to zero
def fill_weights_with_ones(network):
    for name, param in network.named_parameters():
        if 'weight' in name:
            param.data = torch.ones_like(param.data)
        elif 'bias' in name:
            param.data = torch.zeros_like(param.data)
    return network

def compute_receptive_field(network, input_size=(15, 15), binary=True):
    assert isinstance(network, torch.nn.Sequential), 'This only works with torch.nn.Sequential networks.'
    for layer in network:
        if not isinstance(layer, (torch.nn.Conv2d, torch.nn.AvgPool2d)):
            raise Exception('Sorry, this visualisation only works for Conv2d and AvgPool2d.')

    # initialize weights to ones, biases to zeros
    fill_weights_with_ones(network)

    # find the number of input and output channels
    input_channels = None
    output_channels = None
    for layer in network:
        if isinstance(layer, torch.nn.Conv2d):
            if input_channels is None:
                # first convolution layer
                input_channels = layer.in_channels
            output_channels = layer.out_channels
    if input_channels is None:
        input_channels = 1

    # first, we run the forward pass to compute the output shape give the input

    # PyTorch expects input shape [samples, channels, rows, columns]
    x = torch.zeros(1, input_channels, *input_size)
    x.requires_grad = True

    # forward pass: apply each layer in the network
    y = x
    y.retain_grad()
    ys = [y]
    for layer in network:
        y = layer(y)
        # keep track of the intermediate values so we can plot them later
        y.retain_grad()
        ys.append(y)

    # second, we run the backward pass to compute the receptive field

    # create gradient input: zeros everywhere, except for a single pixel
    y_grad = torch.zeros_like(y)
    # put a one somewhere in the middle of the output
    y_grad[0, 0, (y_grad.shape[2] - 1) // 2, (y_grad.shape[3] - 1) // 2] = 1

    # compute the gradients given this single one
    y.backward(y_grad)

    # receptive field is now in the gradient at each layer
    receptive_fields = []
    for y in ys:
        # the gradient for this layer shows us the receptive field
        receptive_field = y.grad
        if binary:
            receptive_field = receptive_field > 0
        receptive_fields.append(receptive_field)
    return receptive_fields

def plot_receptive_field(network, input_size=(15, 15), binary=True):
    receptive_fields = compute_receptive_field(network, input_size, binary)

    # plot the gradient at each layer
    plt.style.use('default')
    plt.figure(figsize=(4 * len(receptive_fields), 4))
    for idx, receptive_field in enumerate(receptive_fields):
        plt.subplot(1, len(receptive_fields), idx + 1)
        # the last element of ys contains the output of the network
        if idx == len(receptive_fields) - 1:
            plot_title = 'output (%dx%d)' % (receptive_field.shape[2], receptive_field.shape[3])
        else:
            plot_title = 'layer %d input (%dx%d)' % (idx, receptive_field.shape[2], receptive_field.shape[3])
        # plot the image with the receptive field (sample 0, channel 0)
        show_image(receptive_field[0, 0], new_figure=False, title=plot_title)
        if not binary:
            plt.colorbar(fraction=0.047 * receptive_field.shape[0] / receptive_field.shape[1])

def receptive_field_size(network, input_size=(15, 15), binary=True):
    receptive_fields = compute_receptive_field(network, input_size, binary)
    return torch.count_nonzero(torch.flatten(receptive_fields[0][0,0]))

Using these functions, we can define a network and plot the receptive field of a pixel in the output.

**(a) Run the code to define a network with one 3×3 convolution layer and plot the images.**

In [ ]:
net = torch.nn.Sequential(
    torch.nn.Conv2d(1, 1, kernel_size=(3, 3)),
)
plot_receptive_field(net, input_size=(15, 15))

Read these images as follows:
* On the left, you see the input size of the network (here: 15 x 15 pixels) and the receptive field for one pixel in the output.
* On the right, you see the output size of the network (here: 13 x 13 pixels).

To visualize the receptive field of this network, we used the following procedure:
* We selected one pixel of the output (shown as the white pixel in the center in the image on the right).
* We computed the gradient for this pixel and plotted the gradient with respect to the input (the image on the left).
* This shows you the receptive field of the network: the output for the pixel we selected depends on these 9 pixels in the input.

**(b) Use this method to plot the receptive field of a pixel in the output of a convolution layer with a kernel size of 5×5.<span style="float:right"> (1 point)</span>**

In [ ]:
# TODO Plot the receptive field of a 5×5 convolution.

net = torch.nn.Sequential(
    torch.nn.Conv2d(1, 1, kernel_size=(5, 5)),
)
plot_receptive_field(net, input_size=(15, 15))



If you look at the result, you will see that two things have changed: the receptive field and the output size.

**(c) How do the receptive field size and the output size depend on the kernel size? Give a formula.<span style="float:right"> (1 point)</span>**

ANSWER:  
The receptive field size has the same number of pixel dimensions as the kernel size the output shrinks according to the formula **input_size - kernel_size +1**.
Receptive field = the size of the region in the input that produces the feature.

SOLUTION:  
The receptive field is the same size as the kernel.
The output size is input_size - kernel_size + 1.

### Counting the number of parameters

In the previous question, you saw how the receptive fields of a 3×3 convolution differs from a 5×5 kernel convolution. But this is not the only difference: there is also a difference in the number of parameters in the network.

We can count the number of parameters in the network by computing the number of elements (e.g., the weights and biases in a convolution kernel) in the parameter list of the PyTorch network.

We'll define a small helper function to do this:

In [ ]:
def num_parameters(network):
    """Count the total number of parameters in a network"""
    return sum([param.data.numel() for param in network.parameters()])

**(d) Use the function to count the number of parameters for a 3×3 convolution.**

In [ ]:
net = torch.nn.Sequential(
    torch.nn.Conv2d(1, 1, kernel_size=(3, 3)),
)
plot_receptive_field(net, input_size=(15, 15))
print(num_parameters(net), "parameters")

**(e) Do the same to count the number of parameters for a 5×5 convolution.<span style="float:right"> (1 point)</span>**

In [ ]:
# TODO count the number of parameters of a 5×5 convolution

net = torch.nn.Sequential(
    torch.nn.Conv2d(1, 1, kernel_size=(5, 5)),
)
plot_receptive_field(net, input_size=(15, 15))
print(num_parameters(net), "parameters")

**(f) Explain the results by showing how to _compute_ the number of parameters for the 3×3 and 5×5 convolutions.<span style="float:right"> (1 point)</span>**

ANSWER:

To compute the number of parameters for the convolution, you just square the size of the convolution kernel (so 3 x 3 or 5 x 5) and then add one parameter for the bias term.

**SOLUTION:**
For k x k kernel, with a input channels and b output channels, thereare a x b x k^2 weights and b biases. For k = 3, and a single channel this gives 3^2 +1 = 10.


For these computations we used convolution layers with one input and one output channel.

We can also compute the results for a layer with a different number of channels.

**(g) Define a network with a 5×5 convolution, 2 input channels and 3 output channels. Print the number of parameters.**

In [ ]:
# TODO count the number of parameters of a 5×5 convolution with 2 input channels and 3 output channels

net = torch.nn.Sequential(
    torch.nn.Conv2d(2, 3, kernel_size=(5, 5)),
)

plot_receptive_field(net, input_size=(15, 15))
print(num_parameters(net), "parameters")

**(h) Show how to compute the number of parameters for this case.<span style="float:right"> (1 point)</span>**

**ANSWER:**  
Parameters are computed by multiplying the kernel size (25) by the number of input channels & by the number of output channels.
Which yields: 25 x 2 x 3 = 150. And then theres a bias term that is being added for each output channel.

### Preserving the size of the input image

The PyTorch documentation for [`torch.nn.Conv2d`](https://pytorch.org/docs/stable/generated/torch.nn.Conv2d.html) describes the parameters that you can use to define a convolutional layer. We will explore some of those parameters in the next questions.

In the previous plot, you may have noticed that the output (13×13 pixels) was slightly smaller than the input (15×15 pixels).

**(i) Define a network with a single 3×3 convolutional layer that produces an output that has the same size as the input.<span style="float:right"> (1 point)</span>**

Use 1 input and 1 output channel.

In [ ]:
# TODO Define a network with a 3×3 kernel size that takes a 15×15 input image
#      and produces a 15×15 output image.

net = torch.nn.Sequential(
    torch.nn.Conv2d(1, 1, kernel_size=(3, 3), padding = (1,)) # add zero-padding to preserve output dimensions
)
plot_receptive_field(net, input_size=(15, 15))
print(num_parameters(net), "parameters")



**(j) Define a network with a single 5×5 convolutional layer that preserves the input size.<span style="float:right"> (1 point)</span>**

In [ ]:
# TODO Define a network with a 5×5 kernel size that takes a 15×15 input image
#      and produces a 15×15 output image.

net = torch.nn.Sequential(
    torch.nn.Conv2d(1, 1, kernel_size=(5, 5), padding = (2,)) # add zero-padding to preserve output dimensions
)
plot_receptive_field(net, input_size=(15, 15))
print(num_parameters(net), "parameters")


Play around with some other values to see how this parameter behaves.

Am I supposed to change the padding parameter, or is there also another way to do it?

Apparently, I can just specify `padding='same'` and then this pads the input so the output has the shape as the input!!

### Multiple layers

As you have just seen, one way to increase the size of the receptive field is to use a larger convolution kernel. But another way is to use more than one convolution layer.

**(k) Define a network with two 3×3 convolutions, preserving the image size. Show the receptive field and the number of parameters.<span style="float:right"> (1 point)</span>**

For this visualisation, do not use any activation functions, and use 1 channel everywhere.

In [ ]:
# TODO define a network with two 3×3 convolutions
net = torch.nn.Sequential(
    torch.nn.Conv2d(1, 1, kernel_size = (3, 3), padding='same'),
    torch.nn.Conv2d(1, 1,  kernel_size = (3, 3), padding='same')
)

print(net)
plot_receptive_field(net, input_size=(15, 15))
print(num_parameters(net), "parameters")

Since we now have two layers, the visualization shows an extra image. From right to left, we have:
* Right: the output size and a single active pixel.
* Middle: the receptive field for the single output pixel between the first and second convolution.
* Left: the receptive field for the single output pixel in the input image.

We have now tried two ways to increase the receptive field size: increasing the kernel size, and using multiple layers.

**(l) Compare the number of parameters required by the two options. Which one is more parameter-efficient?<span style="float:right"> (1 point)</span>**

**ANSWER:**  
Wrong!!
Both options are similar in terms of how many parameters they require. However, increasing the size of the convolution kernel appears to be slightly more efficient.


**SOLUTION:**
The receptive field is the same size for both networks (3x3 + 3x3 and 5x5). The network with two 3x3 kernels has 20 parameters (9 + 1 in two layers), while the 5 x 5 network has 26 parameters (25 +1 in one layer). By stacking two smaller convolutions, we obtain the same receptive field with a smaller number of weights: the deep 3 x 3 network is more parameter-efficient.



**(m) Extra: Construct a network with the same receptive field as in 4.1k, with only 12 parameters.<span style="float:right"> (no points)</span>**

Hint: you don’t have to limit yourself to square kernels.

Not sure how to implement this!

In [ ]:
# TODO define a network with the same receptive field as in k, but with 12 parameters
net = torch.nn.Sequential(
    torch.nn.Conv2d(1, 1, kernel_size=(1, 5), padding=(0, 2)),
    torch.nn.Conv2d(1, 1, kernel_size=(5, 1), padding=(2, 0)),
)


print(net)
plot_receptive_field(net, input_size=(15, 15))
print(num_parameters(net), "parameters")

## 4.2 Variations on convolution (6 points)

### Pooling

We can also increase the size of the receptive field by using a pooling layer.

**(a) Construct a network with a 3×3 convolution (preserving the input size) followed by a 2×2 average pooling. Plot the receptive field and print the number of parameters.<span style="float:right"> (1 point)</span>**

Use 1 input and 1 output channel.

In [ ]:
# TODO define a network with a 3×3 convolution followed by 2×2 average pooling
net = torch.nn.Sequential(
    torch.nn.Conv2d(1, 1, kernel_size = (3, 3), padding='same'),
    torch.nn.AvgPool2d(kernel_size=(2,2))
)

print(net)
plot_receptive_field(net, input_size=(14, 14))
print(num_parameters(net), "parameters")

**(b) Explain the number of parameters in this convolution + pooling network.<span style="float:right"> (1 point)</span>**

ANSWER:  
The pooling operation does not add any new learnable parameter, since it just slides a window over the feature map and computes the average or picks out the maximum value.
So here the total number of parameters just comes from the convolution operation.


### Using strides

By default, convolution layers use a stride of 1.

**(c) Change the network to use a stride of 2 and plot the result.<span style="float:right"> (1 point)</span>**

In [ ]:
# TODO increase the stride to 2
net = torch.nn.Sequential(
    torch.nn.Conv2d(1, 1, kernel_size=(3, 3), padding=(1, 1), stride = 2),
)
print(net)
plot_receptive_field(net, input_size=(14, 14))
print(num_parameters(net), "parameters")


# and one without stride
# TODO increase the stride to 2
net = torch.nn.Sequential(
    torch.nn.Conv2d(1, 1, kernel_size=(3, 3), padding=(1, 1)),
)
print(net)
plot_receptive_field(net, input_size=(14, 14))
print(num_parameters(net), "parameters")

**(d) Explain the new output size and compare the result with that of pooling.<span style="float:right"> (1 point)</span>**

**ANSWER**:  
So with stride = 2 you essentially always shift the convolution kernel by two pixels / points (I think). The new output size is half the layer input size, hence similar to pooling, the output dimensionality gets reduced - except no AvrgPool layer is needed for that.

But the number of parameters is actually the same.

**SOLUTION: **  
Strided convolution has a downsampling effect similar to that of pooling: a stride of 2 reduces the output size by a factor 2.

**(e) Explain how the stride affects the receptive field of this single convolution layer.<span style="float:right"> (1 point)</span>**

**ANSWER:**  
Doubling the stride, halfes the output dimensionality.
Basically, the stride is the *step size* of the kernel and it primarily affects output size.

**SOLUTION:**  
It does not change the receptive field. The network still sees an area of 3x3 pixels when computing the value for an output pixel, it is just skipping over the image in larger steps.


**(f) How does the number of parameters for this network change if we increase the stride?<span style="float:right"> (1 point)</span>**

[a] A network with stride = 2 has twice as many parameters.  
[b] A network with stride = 2 has half as many parameters.  
[c] Changing the stride does not change the number of parameters, it only changes how the convolution is performed.  
[d] The stride and padding cancel each other out, so the number of parameters stays the same.

**ANSWER:**  
[c]

number of learnable parameters in conv layer:    
To calculate the learnable parameters here, all we have to do is just multiply the by the shape of width m, height n, previous layer’s filters d and account for all such filters k in the current layer. Don’t forget the bias term for each of the filter. Number of parameters in a CONV layer would be : ((m * n * d)+1)* k),


**SOLUTION:**   
Strides do not change the number of parameters, it just changes how the convolution moves over the image. The number of parameters is the same as for the earlier 3 x 3 layers.

## 4.3 Padding in very deep networks (2 points)

Without padding, the output of a convolution is smaller than the input. This limits the depth of your network.

**(a) How often can you apply a 3×3 convolution to a 15×15 input image?**

In [ ]:
# find the maximum number of layers
number_of_times = 25

# create a 15×15 input
x = torch.zeros(1, 1, 15, 15)
print('input size: %d×%d' % (x.shape[2], x.shape[3]))

# create a 3×3 convolution
conv = torch.nn.Conv2d(1, 1, kernel_size=(3, 3))

for n in range(number_of_times):
    # apply another convolution
    x = conv(x)
    print('layer %d, output size: %d×%d' % (n + 1, x.shape[2], x.shape[3]))

So I guess the output size always reduces by a factor of kerne_size-1 on each iteration.

Earlier in this assignment, you have used padding to address this problem. This seems ideal.

**(b) Copy the previous code, add some padding, and show that we can now have an infinite number of layers.**

(We are computer scientists and not mathematicians, so for the purpose of this question we'll consider 'infinite' to be equal to 25.)

In [ ]:
# TODO Your code here.

# find the maximum number of layers
number_of_times = 25

# create a 15×15 input
x = torch.zeros(1, 1, 15, 15)
print('input size: %d×%d' % (x.shape[2], x.shape[3]))

# create a 3×3 convolution
conv = torch.nn.Conv2d(1, 1, kernel_size=(3, 3), padding = 'same')

for n in range(number_of_times):
    # apply another convolution
    x = conv(x)
    print('layer %d, output size: %d×%d' % (n + 1, x.shape[2], x.shape[3]))


**(c) Does it really work like this? Have a look at the following experiment.**

* We simulate a convolution network with 25 convolution layers, with 3×3 kernels and the right amount of padding.
* We set the weights to 1/9 (so that the sum of the 3×3 kernel is equal to 1) and set the bias to zero.
* We give this network a 15×15-pixel input filled with ones.
* We plot the output of layers 5, 10, 15, 20, and 25.

In [ ]:
# create a 15×15 input filled with ones
x = torch.ones(1, 1, 15, 15)

# create a 3×3 convolution
conv = torch.nn.Conv2d(1, 1, kernel_size=(3, 3), padding=(1, 1))

# set weights to 1/9 (= sum to one), bias to zero
conv.weight.data = torch.ones_like(conv.weight.data) / 9
conv.bias.data = torch.zeros_like(conv.bias.data)

plt.figure(figsize=(10, 2))
for n in range(1, 26):
    # apply another convolution
    x = conv(x)
    # print('layer %d, output size: %d×%d' % (n + 1, x.shape[2], x.shape[3]))
    if n % 5 == 0:
        plt.subplot(1, 5, n // 5)
        plt.imshow(x[0, 0].detach().numpy(), cmap='gray')
        plt.axis('off')
        plt.title('layer %d' % n)

**(d) Explain the pattern that we see in the output of the final layers. How does this happen, and what does this mean for our very deep networks?<span style="float:right"> (2 points)</span>**

**ANSWER:**  Really not sure what happens here!!
The final classification becomes a lot more blurry I guess and the output is way more pixilated (not sure why this happens though...). So the spatial resolution actually decreased over iterations.


**SOLUTION:**  
What we see in the images are border effects. Padding makes the convolution work outside the image, preserving the image size, but it does so by adding zeros. Since these zeros are different from the pixel or feature values inside the image, the output near the borders will be based on slightly different input values than the pixels in the center of the image.

In the experiment here, we have inputs and kernels filled with constant values. In the center of the image, the output remains 1. At the borders the zeros added by the padding lead to lower values, especially in the late layers.

The effect is more pronounced at late layers of the network: these border artifacts introduced in layer 1 are used as the input for layer 2, where they affect the output a little bit further from the border. This means you cannot have extremely deep networks without introducing these artifacts.


## 4.4 Spoken digits dataset (4 points)

Time for some practical experiments. In the previous assignments, we have used a dataset of images (FashionMNIST), and images are also a common application for CNNs. To mix things up, in this assignment we will investigate CNNs in a completely different domain: speech recognition.

The dataset we use is the free spoken digits dataset, which can be found on https://github.com/Jakobovski/free-spoken-digit-dataset. This dataset consists of the digits 0 to 9, spoken by different speakers. The data comes as .wav files.

**(a) Use the commands below (or a similar tool) to download the dataset. You can also use `git clone` to clone the repository mentioned above.**

In [ ]:
! mkdir -p free-spoken-digit-dataset
! wget -O - https://github.com/Jakobovski/free-spoken-digit-dataset/archive/refs/heads/master.tar.gz | tar xzv -C free-spoken-digit-dataset --strip-components=1

Below is a function to load the data. We pad/truncate each sample to the same length.
The raw audio is usually stored in 16 bit integers, with a range -32768 to 32767, where 0 represents no signal. Before using the data, it should be normalized. A common approach is to make sure that the data is between -1 and 1, or that the data has zero-mean and unit-variance.  Not all of these work well on this data, so later on, if your
network doesn't seem to learn anything: try a different method to see if that works better.

**(b) Update the below code to normalize the data.<span style="float:right"> (1 point)</span>**

Scale the waveform to have mean 0 and variance 1. Note that by doing this per waveform, all files become equally loud.

In [ ]:
sample_rate = 8000
def load_waveform(file, size = 6000):
    sample_rate, waveform = wavfile.read(file)
    # Take first 6000 samples from waveform. With a sample rate of 8000 that corresponds to 3/4 second
    # Pad with 0s if the file is shorter
    waveform = np.pad(waveform,(0,size))[0:size]
    # Normalize waveform
    # TODO: Your code here.

    # not sure if this correct
    waveform = (waveform - waveform.mean()) / waveform.std()
    return waveform

In [ ]:
## correct implementation
    ## BEGIN ANSWER
    # Subtract mean and divide by standard deviation
    waveform = waveform.astype(np.float32)
    waveform -= np.mean(waveform)
    waveform /= np.std(waveform)
    ## END ANSWER
    return waveform


The following code loads all .wav files in a directory, and makes it available in a pytorch dataset.

**(c) Load the data into a variable `data`.**

In [ ]:
class SpokenDigits(torch.utils.data.Dataset):
    def __init__(self, data_dir):
        digits_x = []
        digits_y = []
        for file in os.listdir(data_dir):
            if file.endswith(".wav"):
                waveform = load_waveform(os.path.join(data_dir, file))
                label = int(file[0])
                digits_x.append(waveform)
                digits_y.append(label)
        # convert to torch tensors
        self.x = torch.from_numpy(np.array(digits_x, dtype=np.float32))
        # add an extra dimension to represent the "channels" (we start with 1 channel of data)
        self.x = self.x.unsqueeze(1)
        self.y = torch.from_numpy(np.array(digits_y))

    def __len__(self):
        return len(self.x)

    def __getitem__(self, idx):
        return self.x[idx], self.y[idx]

# TODO: Your code here
data = SpokenDigits(data_dir='/content/free-spoken-digit-dataset/recordings')

# Check if range of values is reasonable
assert abs(torch.mean(data[0][0])) < 1e-4, "Mean of data should be close to 0"
assert torch.max(abs(data[0][0])) < 10, "Data values should not be too large"
assert torch.max(abs(data[0][0])) >= 0.9, "Data values should not be too small"

**(d) Describe the dataset: how many samples are there? How many features does each sample have? How many classes are there?<span style="float:right"> (1 point)</span>**

Note: You may compute the values, or just put in the numeric values.

In [ ]:
# TODO Your answer here.
number_of_samples = data.x.shape[0]  # >> this returns tensor of shape (3000, 1, 6000) - again, I am not sure what these values represent
number_of_features = data.x.shape[2]
number_of_classes = len(torch.unique(data.y))  # should be 10
print('Number of samples:', number_of_samples)
print('Number of features:', number_of_features)
print('Number of classes:', number_of_classes)

Here is code to play samples from the dataset to give you an idea what it "looks" like.

Note: If this step doesn't work in your notebook, then you can ignore it.

In [ ]:
from IPython.display import Audio
def play(sample):
    print(f'Label: {sample[1]}')
    return Audio(sample[0][0].numpy(), rate=sample_rate)
play(data[0])

Before continuing, we split the dataset into a training and a validation set.

In [ ]:
train_fraction = 2/3
train_count = int(len(data) * train_fraction)
train_data, validation_data = torch.utils.data.random_split(data, [train_count, len(data) - train_count])

The code above uses 2/3 of the data for training.

**(e) Discuss an advantage and disadvantage of using more of the data for training.<span style="float:right"> (2 points)</span>**

Advantage of more training data:  
Using more training data, means that the network sees more diverse examples and might therefore generalize better & more robustly to unseen test data.

Disadvantage of more training data:  
You have less data available for validation & tuning of hyperparameters.
But maybe they also want to hear a different answer. Not sure sure...

SOLUTION:
Using more data for training can make the model more accurate.

But if more data is used for training, less remains for testing, and estimates about the generalization ability of the trained model becomes less certain and more noisy


## 4.5 One-dimensional convolutional neural network (8 points)

We will now define a network architecture. We will use a combination of convolutional layers and pooling.
Note that we use 1d convolution and pooling here, instead of the 2d operations used for images.

**(a) Complete the network architecture.<span style="float:right"> (2 points)</span>**

In [ ]:
def build_net():
    return torch.nn.Sequential(
        torch.nn.Conv1d(1, 4, kernel_size=5),
        torch.nn.ReLU(),
        torch.nn.AvgPool1d(kernel_size=2, stride=2),
        # TODO: Add four more convolutional layers, ReLU layers;
        #       doubling the number of channels each time;
        #       Add pooling layers between them.
        # TODO: Your code here.
        # Perform pooling
        torch.nn.Conv1d(4, 8, kernel_size=5),
        torch.nn.ReLU(),
        torch.nn.AvgPool1d(kernel_size=2, stride=2),

        torch.nn.Conv1d(8, 16, kernel_size=5),
        torch.nn.ReLU(),
        torch.nn.AvgPool1d(kernel_size=2, stride=2),

        torch.nn.Conv1d(16, 32, kernel_size=5),
        torch.nn.ReLU(),
        torch.nn.AvgPool1d(kernel_size=2, stride=2),

        torch.nn.Conv1d(32, 64, kernel_size=5),
        torch.nn.ReLU(),
        torch.nn.AvgPool1d(kernel_size=2, stride=2),

        torch.nn.AdaptiveAvgPool1d(100),
        torch.nn.Flatten(),
        torch.nn.Linear(6400, 10)
    )

    # Not sure if I implemented this correctly...

**(b) How many parameters are there in the model? I.e. the total number of weights and biases.<span style="float:right"> (1 point)</span>**

In [ ]:
# TODO: Compute the number of parameters
net = build_net()
print(net)
# plot_receptive_field(net, input_size=(14, 14))
print(num_parameters(net), "parameters")


**(c) The model uses both `AvgPool1d` and `AdapativeAvgPool1d` layers. What is the difference between the two?<span style="float:right"> (1 point)</span>**

**ANSWER:**  

Not sure what the difference is here...   
`AvgPool1d` >>  uses fixed pooling parameters and produces variable output sizes  
`AdaptiveAvgPool1d` >> applies a 1D average pooling over an input signal composed of several input planes. The output size in L_out for any input size. The number of output features is equal to the number of input planes.
- automatically adjusts pooling so that the output has a specific size you choose.

SOLUTION:
AvgPool1d scales the output size by a fixed factor, while AdapativeAvgPool1d scales to a specified output size.

**(d) Suppose that instead of using convolutions, we had used only fully connected layers, while keeping the number of features on each hidden layer the same. How many parameters would be needed in that case approximately?<span style="float:right"> (1 point)</span>**

**ANSWER:**  & SOLUTION:  
Replacing the conv layers with fully connected ones, increases the number of parameters.

The first conv layers has an input of $6000\times 1$ and output of $(6000-4) \times 4$. A fully connected layer with the same input and output size would need $(6000\times 1 + 1) \times (6000-4) \times 4 = 143\,927\,984$ parameters.

Continuing for the other layers, we get a total of $1\,000\,723\,024$ parameters. This is over 600 times more than the convolutional network. Note that each fully connected layer has roughly the same number of parameters, since the input and output halve in size, while the number of input and output channels double. This would not be the case with 2d convolutions.


We will once again need evaluation code and a training loop.
The code below should be familiar from previous assignments.

In [ ]:
def accuracy(pred_y, true_y):
    correct = pred_y.argmax(dim=1) == true_y
    return int(correct.sum()) / len(true_y)

class Metrics:
    """Accumulate mean values of one or more metrics."""
    def __init__(self, n):
        self.count = 0
        self.sum = (0,) * n
    def add(self, count, *values):
        self.count += count
        self.sum = tuple(s + count * v for s,v in zip(self.sum,values))
    def mean(self):
        return tuple(s / self.count for s in self.sum)

def evaluate(net, test_data, batch_size=1000, loss_function=torch.nn.CrossEntropyLoss(), device=device):
    """
    Evaluate a model on the given dataset.
    Return loss, accuracy
    """
    # Note: we can use a large batch size for efficiency, it doesn't matter for the computed loss or accuracy
    test_loader = torch.utils.data.DataLoader(test_data, batch_size=batch_size)
    with torch.no_grad():
        net.eval()
        metrics = Metrics(2)
        for x, y in test_loader:
            x = x.to(device)
            y = y.to(device)
            pred_y = net(x)
            loss = loss_function(pred_y, y)
            acc = accuracy(pred_y, y)
            metrics.add(len(y), loss.item(), acc)
        return metrics.mean()

In [ ]:
class Plotter:
    """For plotting data in animation."""
    # Based on d2l.Animator
    def __init__(self, xlabel=None, ylabel=None, legend=None, xlim=None,
                 ylim=None, xscale='linear', yscale='linear',
                 titles=[],
                 fmts=('-', '--', '-.', ':'), nrows=1, ncols=1,
                 figsize=(5, 3)):
        # Incrementally plot multiple lines
        if legend is None:
            legend = []
        plt.style.use('ggplot')
        self.fig, self.axes = plt.subplots(nrows, ncols, figsize=(figsize[0] * ncols, figsize[1] * nrows))
        if nrows * ncols == 1:
            self.axes = [self.axes, ]
        # Use a function to capture arguments
        def config_axes():
            for axis, title in zip(self.axes, titles):
                axis.set_xlabel(xlabel), axis.set_ylabel(ylabel)
                axis.set_xscale(xscale), axis.set_yscale(yscale)
                axis.set_xlim(xlim),     axis.set_ylim(ylim)
                axis.set_title(title)
                if legend:
                    axis.legend(legend)
        self.config_axes = config_axes
        self.legend = legend
        self.data = [{leg: dict(x=[], y=[], fmt=fmt) for leg,fmt in zip(legend,fmts)} for _ in range(ncols)]

    def add(self, line, x, y):
        if not hasattr(y, "__len__"):
            y = [y]
        if not hasattr(x, "__len__"):
            x = [x] * len(y)
        for a, b, data in zip(x, y, self.data):
            if a is not None and b is not None:
                data[line]['x'].append(a)
                data[line]['y'].append(b)
        self.show()

    def show(self):
        for axis, data in zip(self.axes, self.data):
            axis.cla()
            for line in self.legend:
                line_data = data[line]
                axis.plot(line_data['x'], line_data['y'], line_data['fmt'])
        self.config_axes()
        display.display(self.fig)
        display.clear_output(wait=True)

In [ ]:
def train(net, train_data, validation_data, num_epochs, lr,
          batch_size=100, optimizer=torch.optim.Adam, device=device):
    """
    Train a network on the given training data.
    After every epoch compute validation loss and accuracy.
    """
    net.to(device)
    train_loader = torch.utils.data.DataLoader(train_data, batch_size=batch_size, shuffle=True)
    num_batches = len(train_loader)
    optimizer = optimizer(net.parameters(), lr=lr)
    loss_function = torch.nn.CrossEntropyLoss()
    plotter = Plotter(xlabel='epoch', xlim=[1, num_epochs], ncols=2,
                      titles=['loss','accuracy'], legend=['train','validation'])
    start_time = time.time()
    for epoch in range(num_epochs):
        # Sum of training loss, sum of training accuracy, no. of examples
        net.train()
        metrics = Metrics(2)
        for i, (x, y) in enumerate(train_loader):
            #timer.start()
            optimizer.zero_grad()
            x = x.to(device)
            y = y.to(device)
            pred_y = net(x)
            loss = loss_function(pred_y, y)
            loss.backward()
            optimizer.step()
            with torch.no_grad():
                acc = accuracy(pred_y, y)
                metrics.add(len(y), loss.item(), acc)
            if (i + 1) % (num_batches // 5) == 0 or i == num_batches - 1:
                train_loss, train_acc = metrics.mean()
                plotter.add('train', epoch + (i + 1) / num_batches, (train_loss, train_acc))

        val_loss, val_acc = evaluate(net, validation_data, loss_function=loss_function, device=device)
        plotter.add('validation', epoch + 1, (val_loss, val_acc))

    train_loss, train_acc = metrics.mean()
    train_time = time.time() - start_time
    print(f'train loss {train_loss:.3f}, train acc {train_acc:.3f}, '
          f'val loss {val_loss:.3f}, val acc {val_acc:.3f}')
    print(f'{metrics.count * num_epochs / train_time:.1f} samples/sec '
          f'on {str(device)}')

The FashionMNIST dataset that we used before has 60000 training examples, of which we used only 1000.
When using the entire FashionMNIST dataset, around 10 epochs are needed to train a convolutional neural network.
How large is our training set this time? How would this affect the number of epochs that we need?

**(e) How many epochs do you think are needed?<span style="float:right"> (1 point)</span>**

In [ ]:
lr, num_epochs = 0.001, 20 # TODO: change

# our training set this time is a lot smaller, so we would probably need a lot more epochs for the training to converge. Maybe around 100 epochs.

**(f) Now train the network.**

In [ ]:
torch.manual_seed(42) # Fix the seed, so outputs are exactly reproducible
train(build_net(), train_data, validation_data, num_epochs=num_epochs, lr=lr)

**(g) Did the training converge?<span style="float:right"> (2 point)</span>**

**If the training has not converged, maybe you need to change the number of epochs and/or the learning rate.**

Hint: This is a non-trivial problem, so your network might take some time to
learn. Don't give up too quickly, it might take 50-100 epochs before you
see any significant changes in the loss curves.

TODO: Document the runs that you have performed and thir results in the table below.

| Experiment                | epochs | lr     | train accuracy | val. accuracy | converged? |
|---------------------------|--------|--------|----------------|---------------|------------|
| experiment 1              | 100  | 0.001   |       1.00 | 0.881        |           yes
| experiment 2 | 100 | 0.01 | 1.00 | 0.894 |kind of |
| experiment 3 | 20 | 0.001 | 0.894 | 0.806 | no

## 4.6 Questions and evaluation (6 points)

**(a) Does the network look like it is overfitting or underfitting? Explain how you see this.<span style="float:right"> (1 point)</span>**

ANSWER:  
It looks like the network is overfitting on the training set (when epoch number is set to 100), because there the accuracy is perfect and the loss very low, whereas the loss on the validation set is steadily increasing. With less epochs (i.e. 20), the training hasn't fully converged yet - but there is a better balance between overfitting and underfitting.

**(b) Is what we have here a good classifier? Could it be used in a realistic application? Motivate your answer.<span style="float:right"> (1 point)</span>**

ANSWER:
No, for a good classifier, we would expect better out-of-distribution generalization and a lower loss on the validation set.

**SOLUTION:**  
The accuracy on new (test) data is around 0.85. While this is better than chance, making an error in 15% of the cases is not usable in practical applications. Imagine dictating a ten digit phone number. On average the network would classify 1 or 2 digits wrong.



**(c) Do you think there is enough training data compared to the dimensions of the data and the number of parameters? Motivate your answer.<span style="float:right"> (1 point)</span>**

ANSWER:  
No, I don't think there is enough training data, especially for close to 70000 parameters.


**SOLUTION:**   
The network has over 70.000 parameters, which is much more than the number of data points. With strong regularization it is possible to train networks with more parameters than training points, but they still need a lot of data.

Another issue is the length of input, since the input has 6000 features, there are also more features than samples. In such a setting even training a linear modelis hard.

**(d) How could the classifier be improved? Give at least 2 suggestions.<span style="float:right"> (1 point)</span>**

ANSWER:  
The classifier performance could be improved by including batch normalization of the feature maps or by introducing skip connections into the network architecture.

SOLUTION - Some ways to improve the classifier are:
- adding regularization, for instance with weight decay or dropout
- tweaking the network architecture (decreasing the number of layers, experimenting with kernel size of the convolutions)
- using more training data
- use data augmentation
- use a pre-trained or manual feature detector



**(e) The free spoken digits datasets has recordings from several different speakers. Is the validation set accuracy a good measure of how well the trained network would perform for recognizing digits spoken by a new, unknown speaker? And if not, how could that be tested instead?<span style="float:right"> (2 points)</span>**

**SOLUTION:**   
Because the train/validation split is random, all speakers in the validation set are also very likely to be in the training set. So this is not representative of the case of recognizing a new speaker.

To test whether the network can recognize new speakers, a different train/validation split can be used where speakers are divided between the two sets instead of the individual samples. In that way none of the speakers in the test set occur in the training set.

Because the number of speakers is quite small, the test set accuracy is still not guaranteed to give a good estimate of the generalization performance. It can be helpful to do this multiple times with cross-validation.

## 4.7 Variations (8 points)

One way in which the training might be improved is with dropout or with batch normalization.

**(a) Make a copy of the network architecture from 4.5a below, and add dropout after every convolutional layer.<span style="float:right"> (1 point)</span>**

In [ ]:
def build_net_dropout(dropout = 0.3):
  return torch.nn.Sequential(
        torch.nn.Conv1d(1, 4, kernel_size=5),
        torch.nn.Dropout1d(p = dropout),
        torch.nn.ReLU(),
        torch.nn.AvgPool1d(kernel_size=2, stride=2),

        torch.nn.Conv1d(4, 8, kernel_size=5),
        torch.nn.Dropout1d(p = dropout),

        torch.nn.ReLU(),
        torch.nn.AvgPool1d(kernel_size=2, stride=2),

        torch.nn.Conv1d(8, 16, kernel_size=5),
        torch.nn.Dropout1d(p = dropout),
        torch.nn.ReLU(),
        torch.nn.AvgPool1d(kernel_size=2, stride=2),

        torch.nn.Conv1d(16, 32, kernel_size=5),
        torch.nn.Dropout1d(p = dropout),
        torch.nn.ReLU(),
        torch.nn.AvgPool1d(kernel_size=2, stride=2),

        torch.nn.Conv1d(32, 64, kernel_size=5),
        torch.nn.Dropout1d(p = dropout),
        torch.nn.ReLU(),
        torch.nn.AvgPool1d(kernel_size=2, stride=2),

        torch.nn.AdaptiveAvgPool1d(100),
        torch.nn.Flatten(),
        torch.nn.Linear(6400, 10)
    )

lr, num_epochs = 0.001, 50

torch.manual_seed(42) # Fix the seed, so outputs are exactly reproducible
train(build_net_dropout(), train_data, validation_data, num_epochs=num_epochs*3//2, lr=lr)

**(b) How does dropout change the results? Does this match what you saw on the simple network last week?<span style="float:right"> (1 point)</span>**

**ANSWER:**   
Dropout makes the training process more noisy (less smooth), and more slow. But it exerts a regulating effect on network training such that performance curves of train and validation set are way closer together, indicating better generalization performance.


**SOLUTION:**  
By introducing dropout, the network becomes slower to converge, and there is more variation in the accuracy and loss.
But the final validation accuracy is higher while the train accuracy is lower. This means that we have reduced the amount of overfitting.

**(c) Make a copy of the original network architecture, and add batch normalization to all convolutional and linear layers.<span style="float:right"> (1 point)</span>**

In [ ]:
def build_net_batchnorm():
        return torch.nn.Sequential(
        torch.nn.Conv1d(1, 4, kernel_size=5),
        torch.nn.BatchNorm1d(4),
        torch.nn.ReLU(),
        torch.nn.AvgPool1d(kernel_size=2, stride=2),

        torch.nn.Conv1d(4, 8, kernel_size=5),
        torch.nn.BatchNorm1d(8),
        torch.nn.ReLU(),
        torch.nn.AvgPool1d(kernel_size=2, stride=2),

        torch.nn.Conv1d(8, 16, kernel_size=5),
        torch.nn.BatchNorm1d(16),
        torch.nn.ReLU(),
        torch.nn.AvgPool1d(kernel_size=2, stride=2),

        torch.nn.Conv1d(16, 32, kernel_size=5),
        torch.nn.BatchNorm1d(32),
        torch.nn.ReLU(),
        torch.nn.AvgPool1d(kernel_size=2, stride=2),

        torch.nn.Conv1d(32, 64, kernel_size=5),
        torch.nn.BatchNorm1d(64),
        torch.nn.ReLU(),
        torch.nn.AvgPool1d(kernel_size=2, stride=2),

        torch.nn.AdaptiveAvgPool1d(100),
        torch.nn.Flatten(),
        torch.nn.Linear(6400, 10),
      #  torch.nn.BatchNorm1d(10) # !! don't use BatchNorm after the final layer

    )

torch.manual_seed(42) # Fix the seed, so outputs are exactly reproducible
train(build_net_batchnorm(), train_data, validation_data, num_epochs=num_epochs, lr=lr)

**(d) How does batch normalization change the results? Does this match what you saw on the simple network last week?<span style="float:right"> (1 point)</span>**

**ANSWER:**  
Batch Normalization stabilizes the learning process and leads to faster learning, we can see here with the CNN (similar to last weeks simple network), that accuracy and loss curves are way steeper and training a CNN with batch norm achieves faster convergence.

**SOLUTION:**  
The network with batch normalization converges much faster than the baseline network without batch normalization, and it reaches a higher validation accuracy.

### Residual network

We can also try to use a residual network.

Pytorch does not expose a general purpose block for residual connections.
If you look at the [source code for ResNet vision model](https://pytorch.org/vision/0.8/_modules/torchvision/models/resnet.html), you will find the following definitions (slightly simplified):

In [ ]:
# From https://pytorch.org/vision/0.8/_modules/torchvision/models/resnet.html
# (simplified)

def conv3x3(in_channels, out_channels, stride=1):
    """3x3 convolution with padding"""
    return torch.nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, stride=stride, bias=False)

def conv1x1(in_channels, out_channels, stride=1):
    """1x1 convolution, used for downsampling"""
    return torch.nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False)

class ResidualBlock2d(torch.nn.Module):
    def __init__(self, in_channels, channels, stride=1, downsample=None, norm_layer=None):
        super(ResidualBlock2d, self).__init__()
        if norm_layer is None:
            norm_layer = torch.nn.BatchNorm2d
        # Both self.conv1 and self.downsample layers downsample the input when stride != 1
        self.conv1 = conv3x3(in_channels, channels, stride)
        self.bn1 = norm_layer(channels)
        self.relu = torch.nn.ReLU(inplace=True)
        self.conv2 = conv3x3(channels, channels)
        self.bn2 = norm_layer(channels)
        if downsample is True:
            downsample = torch.nn.Sequential(conv1x1(in_channels, channels), norm_layer(channels))
        self.downsample = downsample
        self.stride = stride

    def forward(self, x):
        identity = x

        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)

        out = self.conv2(out)
        out = self.bn2(out)

        if self.downsample is not None:
            identity = self.downsample(x)

        out += identity
        out = self.relu(out)

        return out

**(e) Copy the `ResidualBlock2d` module below, and adapt it for 1d convolutions. Use a kernel size of 5 for the convolution layers.<span style="float:right"> (2 points)</span>**

Use residual blocks each containing two convolutional layers.

In [ ]:
# TODO: Residual class here

def conv5x5(in_channels, out_channels, stride=1):
    """5x5 convolution with padding"""
    return torch.nn.Conv1d(in_channels, out_channels, kernel_size=5, padding="same", stride=stride, bias=False)

def conv1x1(in_channels, out_channels, stride=1):
    """1x1 convolution, used for downsampling"""
    return torch.nn.Conv1d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False)

class ResidualBlock1d(torch.nn.Module):
    def __init__(self, in_channels, channels, stride=1, downsample=None, norm_layer=None):
        super(ResidualBlock1d, self).__init__()
        if norm_layer is None:
            norm_layer = torch.nn.BatchNorm1d
        # Both self.conv1 and self.downsample layers downsample the input when stride != 1
        self.conv1 = conv5x5(in_channels, channels, stride)
        self.bn1 = norm_layer(channels)
        self.relu = torch.nn.ReLU(inplace=True)
        self.conv2 = conv5x5(channels, channels)
        self.bn2 = norm_layer(channels)
        if downsample is True:
            downsample = torch.nn.Sequential(conv1x1(in_channels, channels), norm_layer(channels))
        self.downsample = downsample
        self.stride = stride

    def forward(self, x):
        identity = x

        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)

        out = self.conv2(out)
        out = self.bn2(out)

        if self.downsample is not None:
            identity = self.downsample(x)

        out += identity
        out = self.relu(out)

        return out



**(f) Make a copy of the network architecture from 4.5a, and replace all convolutions with residual blocks.<span style="float:right"> (1 point)</span>**

In [ ]:
def build_resnet():
    norm_layer = torch.nn.BatchNorm1d # Define the normalization layer
    return torch.nn.Sequential(

        ResidualBlock1d(1, 4, norm_layer=norm_layer, downsample=True),

        torch.nn.AvgPool1d(kernel_size=2, stride=2),


        ResidualBlock1d(4, 8, norm_layer=norm_layer, downsample=True),

        torch.nn.AvgPool1d(kernel_size=2, stride=2),

        ResidualBlock1d(8, 16, norm_layer=norm_layer, downsample=True),

        torch.nn.AvgPool1d(kernel_size=2, stride=2),

        ResidualBlock1d(16, 32, norm_layer=norm_layer, downsample=True),

        torch.nn.AvgPool1d(kernel_size=2, stride=2),


        ResidualBlock1d(32, 64, norm_layer=norm_layer, downsample=True),

        torch.nn.AvgPool1d(kernel_size=2, stride=2),

        torch.nn.AdaptiveAvgPool1d(100),
        torch.nn.Flatten(),
        torch.nn.Linear(6400, 10)
    )

torch.manual_seed(42)
train(build_resnet(), train_data, validation_data, num_epochs, lr = 0.001)

**(g) How do residual connections change the results?<span style="float:right"> (1 point)</span>**

**SOLUTION:**  
- The network is slower to train (in terms of wall clock time), each epoch takes 1.5-2 times as long. This happens because the number of convolution layers is larger, because each residual block contains 2 conv5 layers (and a conv1 layer which is fast)
- The model converges faster, and to a very low training loss
- The validation accuracy is higher than before. This is probably a result of having a more complex deeper network
- There might be some weird peeks in the training and test loss. This happens because the Adam optimizer starts to make steps that are too large as the loss becomes small.

## 4.8 A ConvNet for the 2020s (10 points)

There have been several developments in convolutional networks in the last 10 years, and the networks we have used so far in this assingment do not reflect the latest advancements.
A good paper that describes the more up-to-date designs is [*A ConvNet for the 2020s*, by Z. Liu, H. Mao, C. Wu, C. Feichtenhofer, T. Darrell, and S. Xie](https://arxiv.org/abs/2201.03545), which updates CNNs with the lessons learned from vision transformer networks (which we will talk about in week 6).

Not all of these ideas apply to the 1d data that we are using here, but some do.

**(a) Have a *brief* look at [the paper](https://arxiv.org/abs/2201.03545).**

Don't try to read everything in detail, instead, read the abstract, scroll through the pdf file, and look at the figures.

The starting point of their paper is a residual block with 3 convolutional layers, where the first and last have kernel size 1. This differs a bit from the residual block from the previous section.

**(b) Copy the residual block from 4.7e, add an extra convolutional layer and adjust the kernel sizes to match the ResNet block of the paper (figure 4).<span style="float:right"> (1 point)</span>**

Keep the number of channels the same as in 4.7e.

In [ ]:
# solution:

### BEGIN ANSWER
def conv5(in_channels, out_channels, stride=1):
    """size 5 convolution with padding"""
    return torch.nn.Conv1d(in_channels, out_channels, kernel_size=5, padding=2, stride=stride, bias=False)

def conv1(in_channels, out_channels, stride=1):
    """size 1 convolution"""
    return torch.nn.Conv1d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False)

class BaselineResidualBlock1d(torch.nn.Module):
    def __init__(self, in_channels, channels, stride=1, downsample=None, norm_layer=None):
        super(BaselineResidualBlock1d, self).__init__()
        if norm_layer is None:
            norm_layer = torch.nn.BatchNorm1d
        self.conv1 = conv1(in_channels, channels, stride)
        self.bn1 = norm_layer(channels)
        self.relu = torch.nn.ReLU(inplace=True)
        self.conv2 = conv5(channels, channels)
        self.bn2 = norm_layer(channels)
        self.conv3 = conv1(channels, channels)
        self.bn3 = norm_layer(channels)
        if downsample is True:
            downsample = torch.nn.Sequential(conv1(in_channels, channels), norm_layer(channels))
        self.downsample = downsample
        self.stride = stride

    def forward(self, x):
        identity = x

        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)
        out = self.conv2(out)
        out = self.bn2(out)
        out = self.relu(out)
        out = self.conv3(out)
        out = self.bn3(out)

        if self.downsample is not None:
            identity = self.downsample(x)

        out += identity
        out = self.relu(out)

        return out
### END ANSWER

In [ ]:
# Optional: train a resnet with this baseline architecture

## BEGIN ANSWER
def build_baseline_resnet():
    return torch.nn.Sequential(
        BaselineResidualBlock1d(1, 4, downsample=True),
        torch.nn.AvgPool1d(kernel_size=2, stride=2),
        BaselineResidualBlock1d(4, 8, downsample=True),
        torch.nn.AvgPool1d(kernel_size=2, stride=2),
        BaselineResidualBlock1d(8, 16, downsample=True),
        torch.nn.AvgPool1d(kernel_size=2, stride=2),
        BaselineResidualBlock1d(16, 32, downsample=True),
        torch.nn.AvgPool1d(kernel_size=2, stride=2),
        BaselineResidualBlock1d(32, 64, downsample=True),
        torch.nn.AdaptiveAvgPool1d(100),
        torch.nn.Flatten(),
        torch.nn.Linear(6400, 10)
    )

torch.manual_seed(42) # Fix the seed, so outputs are exactly reproducible
train(build_baseline_resnet(), train_data, validation_data, num_epochs=num_epochs, lr=lr)
## END ANSWER

**(c) We have added an extra layer to each resnet block. Does this increase the number of parameters in the ResNet block? Briefly explain your answer.<span style="float:right"> (1 point)</span>**

Hint: you can compute this.

In [ ]:
# Solution:

### BEGIN ANSWER
print('previous resnet model', num_parameters(ResidualBlock1d(10, 10)))
print('new "baseline" model ', num_parameters(BaselineResidualBlock1d(10, 10)))
# Note: number of channels doesn't matter for the conclusion as long as it is > 1.
### END ANSWER

**SOLUTION:**  
The number of parameters becomes smaller, because a 5x5 convolution layer has 25 times more parameters than a 1x1 convolution. In comparison the two 1x1 convolution layers are insignificant.

In section 2.3, the authors propose to use depthwise convolution.

**(d) What is an advantage and a disadvantage of depthwise over normal convolutions, all else being equal?<span style="float:right"> (1 point)</span>**

Advantage:
less multiplications are required, which in turn reduces the number of network parameters.

Disadvantage:  
channel information will not be combibed, hence they might not pick up cross-correlations across channels.


**SOLUTION:**  
Answer:

Advantage:

fewer FLOPs (faster), this is mentioned in the paper.
fewer parameters (reduces overfitting)
Disadvantage:

lower accuracy, this is mentioned in the paper.

**(e) Implement a 1d depthwise convolution.<span style="float:right"> (1 point)</span>**

You may assume that the number of output channels is a multiple of the number of input channels.

If you need a hint, have a look at [the torch documentation for Conv1D](https://docs.pytorch.org/docs/stable/generated/torch.nn.Conv1d.html).

In [ ]:
def conv_depthwise(in_channels, out_channels, kernel_size=5, stride=1):
    """
    Depthwise convolution with the given kernel size, with padding.
    """
    # TODO: implement this function

    ### BEGIN ANSWER
    return torch.nn.Conv1d(in_channels, out_channels,
                           kernel_size=kernel_size, padding=kernel_size//2, stride=stride, bias=False,
                           groups=in_channels)
    ### END ANSWER

Section 2.6 of the paper describes several changes to the basic block. These are also shown in figure 4.

**(f) Implement a 1 dimensional version of the ConvNeXtBlock.<span style="float:right"> (3 points)</span>**

Keep the number of channels the same as in the previous questions.

In [ ]:
## SOLUTION

### BEGIN ANSWER
class ConvNeXtBlock1d(torch.nn.Module):
    def __init__(self, in_channels, channels, stride=1, downsample=None, norm_layer=None):
        super(ConvNeXtBlock1d, self).__init__()
        if norm_layer is None:
            norm_layer = LayerNorm1d
        self.conv1 = conv_depthwise(in_channels, channels, kernel_size=7, stride=stride)
        self.ln1 = norm_layer(channels)
        self.gelu = torch.nn.GELU()
        self.conv2 = conv1(channels, channels * 4)
        self.conv3 = conv1(channels * 4, channels)
        if downsample is True:
            downsample = torch.nn.Sequential(conv1(in_channels, channels), norm_layer(channels))
        self.downsample = downsample
        self.stride = stride

    def forward(self, x):
        identity = x

        out = self.conv1(x)
        out = self.ln1(out)
        out = self.conv2(out)
        out = self.gelu(out)
        out = self.conv3(out)

        if self.downsample is not None:
            identity = self.downsample(x)

        out += identity

        return out
### END ANSWER

# You may need this
class LayerNorm1d(torch.nn.LayerNorm):
    """1d layer normalization"""
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x.permute(0, 2, 1)
        x = torch.nn.functional.layer_norm(x, self.normalized_shape, self.weight, self.bias, self.eps)
        x = x.permute(0, 2, 1)
        return x

**(g) Define and train a model with 1d ResNeXt blocks.<span style="float:right"> (no points)</span>**

Use the same architecture as in previous experiments, except with ResNeXt blocks instead of (baseline) ResNet blocks.

In [ ]:
## SOLUTION
def build_resnext():
    return torch.nn.Sequential(
        ConvNeXtBlock1d(1, 4, downsample=True),
        torch.nn.AvgPool1d(kernel_size=2, stride=2),
        ConvNeXtBlock1d(4, 8, downsample=True),
        torch.nn.AvgPool1d(kernel_size=2, stride=2),
        ConvNeXtBlock1d(8, 16, downsample=True),
        torch.nn.AvgPool1d(kernel_size=2, stride=2),
        ConvNeXtBlock1d(16, 32, downsample=True),
        torch.nn.AvgPool1d(kernel_size=2, stride=2),
        ConvNeXtBlock1d(32, 64, downsample=True),
        torch.nn.AdaptiveAvgPool1d(100),
        torch.nn.Flatten(),
        torch.nn.Linear(6400, 10)
    )

torch.manual_seed(42) # Fix the seed, so outputs are exactly reproducible
train(build_resnext(), train_data, validation_data, num_epochs=num_epochs, lr=lr)
## END ANSWER

**(h) How do the results of this ConvNeXt-like model compare to the ResNet from the previous section?<span style="float:right"> (1 point)</span>**

**SOLUTION:**  
The ResNeXt model performs worse in terms of validation loss and validation accuracy. It looks like the model is overfitting more.

**(i) We are still missing some changes that are made in ConvNeXt, that could be applied to our architecture as well. Give at least two.<span style="float:right"> (2 points)</span>**

**SOLUTION:**  
The number of channels stays the same between blocks.
The number of channels is larger (96 in the paper).
Pooling/downsampling is performed using stride 2 convolutions
The first layer 'patchifies' the input, using a convolution with stride=kernel_size=4

## 4.9 Feature extraction (5 points)

Given enough training data a deep neural network can learn to extract features from raw data like audio and images. However, in some cases it is still necessary to do manual feature extraction, in particular when working with smaller datasets like this one. For speech recognition, a popular class of features are [MFCCs](https://en.wikipedia.org/wiki/Mel-frequency_cepstrum).

Here is code to extract these features. You will need to install the `python_speech_features` first.

In [ ]:
!pip install python_speech_features

from python_speech_features import mfcc

def load_waveform_mfcc(file, size = 6000):
    sample_rate, waveform = wavfile.read(file)
    waveform = np.pad(waveform,(0,size))[0:size] / 32768
    return np.transpose(mfcc(waveform, sample_rate))

**(a) Implement a variation of the dataset that uses these features.<span style="float:right"> (2 points)</span>**

In [ ]:
class SpokenDigitsMFCC(torch.utils.data.Dataset):
    ## BEGIN ANSWER
    def __init__(self, data_dir):
        digits_x = []
        digits_y = []
        for file in os.listdir(data_dir):
            if file.endswith(".wav"):
                waveform = load_waveform_mfcc(os.path.join(data_dir, file))
                label = int(file[0])
                digits_x.append(waveform)
                digits_y.append(label)
        # convert to torch tensors
        self.x = torch.from_numpy(np.array(digits_x, dtype=np.float32))
        self.y = torch.from_numpy(np.array(digits_y))
    def __len__(self):
        return len(self.x)
    def __getitem__(self, idx):
        return self.x[idx], self.y[idx]
    ## END ANSWER
data_mfcc = SpokenDigitsMFCC(data_dir) # TODO: your data directory here
train_count_mfcc = int(len(data_mfcc) * train_fraction)
train_data_mfcc, validation_data_mfcc = torch.utils.data.random_split(data_mfcc, [train_count_mfcc, len(data_mfcc)-train_count_mfcc])

assert train_data_mfcc[0][0].shape == torch.Size([13,74]), "There is something wrong with the SpokenDigitsMFCC dataset"

The MFCC features will have 13 channels instead of 1 (the `unsqueeze` operation is not needed).

**(b) Inspect the shape of the data, and define a new network architecture that accepts data with this shape.<span style="float:right"> (1 point)</span>**

Note: you might want to use fewer layers than in the 1d network.

In [ ]:
## BEGIN ANSWER
number_of_samples = len(data_mfcc)
data_shape = data_mfcc[0][0].shape
number_of_classes = len(np.unique([y for (x,y) in data_mfcc]))
print('Number of samples:', number_of_samples)
print('Shape of samples:', data_shape)
print('Number of classes:', number_of_classes)


def build_net_mfcc():
  return torch.nn.Sequential(
        # there are now 13 input channels instead of 1
        torch.nn.Conv1d(13, 16, kernel_size=5),
        torch.nn.ReLU(),
        torch.nn.AvgPool1d(kernel_size=2, stride=2),
        torch.nn.Conv1d(16, 32, kernel_size=5),
        torch.nn.ReLU(),
        torch.nn.AvgPool1d(kernel_size=2, stride=2),
        torch.nn.Conv1d(32, 64, kernel_size=5),
        torch.nn.ReLU(),
        torch.nn.AvgPool1d(kernel_size=2, stride=2),
        torch.nn.Conv1d(64, 128, kernel_size=5),
        torch.nn.ReLU(),
        # the samples are shorter (lower temporal resolution), so coarser pooling makes sense here
        torch.nn.AdaptiveAvgPool1d(5),
        torch.nn.Flatten(),
        torch.nn.Linear(128*5, 10)
    )
## END ANSWER


**(c) Train the network with the MFCC features.<span style="float:right"> (1 point)</span>**

In [ ]:
## BEGIN ANSWER
torch.manual_seed(42) # Fix the seed, so outputs are exactly reproducible
train(build_net_mfcc(), train_data_mfcc, validation_data_mfcc, num_epochs=20, lr=0.001)
## END ANSWER

**(d) What would be needed to get a fully neural network approach to work as well as MFCC features?<span style="float:right"> (1 point)</span>**

**SOLUTION:**  
- A neural network needs enough training data to compete with MFCC features. If another dataset is available, a network could be pretrained on that other dataset. If only unlabeled data is available an unsupervised pre-training method such as wav2vec can be used
- You can also see that the residual network performed quite well, so perhaps a change in architecture (increasing the complexity of the network) might also improve the results

## The end

Well done! Please double check the instructions at the top before you submit your results.

*This assignment has 58 points.*
<span style="float:right;color:#aaa;font-size:10px;"> Version 7df5e92 / 2025-11-26</span>